<a href="https://colab.research.google.com/github/RA-1020/tiny-wfm-backbone/blob/main/phase0_shared_part_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!pip install timm wandb -q

Fri Jul  3 14:11:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
import torch.nn as nn
import timm
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import wandb
wandb.login(key="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")

device = torch.device("cuda")
print("Using:", torch.cuda.get_device_name(0))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ra-1020 (ra-10) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using: Tesla T4


In [4]:
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),                      # image -> tensor, values 0..1
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],   # CIFAR-10 channel means
                std=[0.2470, 0.2435, 0.2616]),   # and standard deviations
])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Subsample for speed: 10k train / 2k test is plenty to prove learning
train_set = torch.utils.data.Subset(train_set, range(10_000))
test_set  = torch.utils.data.Subset(test_set,  range(2_000))

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False, num_workers=2)

print(len(train_set), "train /", len(test_set), "test")

100%|██████████| 170M/170M [01:42<00:00, 1.67MB/s]


10000 train / 2000 test


In [5]:
model = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=False,
    num_classes=10,
    in_chans=3,         # RGB; becomes 1 for your spectrograms later
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.2f}M parameters")   # ~5.7M

5.53M parameters


In [6]:
EPOCHS = 5
LR = 1e-4   # note: 1e-4, not 1e-3 — ViTs from scratch are unstable at high LR

criterion = nn.CrossEntropyLoss()                      # the "how wrong" measure
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

run = wandb.init(project="tiny-wfm-backbone", name="stage-b-vit-cifar10",
                 config={"model": "vit_tiny_patch16_224", "lr": LR,
                         "epochs": EPOCHS, "train_size": len(train_set)})

def evaluate():
    model.eval()                       # eval mode (disables dropout etc.)
    correct = total = 0
    with torch.no_grad():              # no gradients needed for testing
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(EPOCHS):
    model.train()
    running_loss = correct = total = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)              # 1. forward pass
        loss = criterion(logits, y)    # 2. loss

        optimizer.zero_grad()          #    clear old gradients
        loss.backward()                # 3. backward pass
        optimizer.step()               # 4. nudge the parameters

        running_loss += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total
    val_acc    = evaluate()
    wandb.log({"epoch": epoch, "train_loss": train_loss,
               "train_acc": train_acc, "val_acc": val_acc})
    print(f"epoch {epoch}: loss {train_loss:.3f} | train {train_acc:.1%} | val {val_acc:.1%}")

wandb.finish()

epoch 0: loss 1.969 | train 26.0% | val 31.1%
epoch 1: loss 1.805 | train 32.5% | val 34.1%
epoch 2: loss 1.720 | train 35.6% | val 36.5%
epoch 3: loss 1.648 | train 38.6% | val 39.5%
epoch 4: loss 1.590 | train 41.4% | val 43.0%


epoch,▁▃▅▆█
train_acc,▁▄▅▇█
train_loss,█▅▃▂▁
val_acc,▁▃▄▆█
epoch,4
train_acc,0.4143
train_loss,1.59042
val_acc,0.4295


In [9]:
EPOCHS = 15
LR = 1e-4

model = timm.create_model("vit_tiny_patch16_224", pretrained=False,
                          num_classes=10, in_chans=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

run = wandb.init(project="tiny-wfm-backbone", name="stage-b-vit-cifar10-15ep",
                 config={"model": "vit_tiny_patch16_224", "lr": LR,
                         "epochs": EPOCHS, "train_size": len(train_set)})

# same evaluate() + training loop as before, just re-run with new EPOCHS

def evaluate():
    model.eval()                       # eval mode (disables dropout etc.)
    correct = total = 0
    with torch.no_grad():              # no gradients needed for testing
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(EPOCHS):
    model.train()
    running_loss = correct = total = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)              # 1. forward pass
        loss = criterion(logits, y)    # 2. loss

        optimizer.zero_grad()          #    clear old gradients
        loss.backward()                # 3. backward pass
        optimizer.step()               # 4. nudge the parameters

        running_loss += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total
    val_acc    = evaluate()
    wandb.log({"epoch": epoch, "train_loss": train_loss,
               "train_acc": train_acc, "val_acc": val_acc})
    print(f"epoch {epoch}: loss {train_loss:.3f} | train {train_acc:.1%} | val {val_acc:.1%}")

wandb.finish()

epoch 0: loss 1.984 | train 25.5% | val 28.4%
epoch 1: loss 1.801 | train 32.8% | val 34.6%
epoch 2: loss 1.686 | train 37.3% | val 39.4%
epoch 3: loss 1.609 | train 40.5% | val 40.6%
epoch 4: loss 1.538 | train 44.4% | val 43.2%
epoch 5: loss 1.474 | train 46.7% | val 46.0%
epoch 6: loss 1.425 | train 47.7% | val 46.0%
epoch 7: loss 1.403 | train 48.8% | val 47.7%
epoch 8: loss 1.341 | train 51.0% | val 45.1%
epoch 9: loss 1.296 | train 52.6% | val 49.8%
epoch 10: loss 1.259 | train 53.9% | val 48.5%
epoch 11: loss 1.223 | train 55.8% | val 52.2%
epoch 12: loss 1.185 | train 57.6% | val 51.8%
epoch 13: loss 1.155 | train 58.1% | val 51.0%
epoch 14: loss 1.121 | train 59.1% | val 52.0%


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▃▃▄▅▅▆▆▆▇▇▇███
train_loss,█▇▆▅▄▄▃▃▃▂▂▂▂▁▁
val_acc,▁▃▄▅▅▆▆▇▆▇▇████
epoch,14
train_acc,0.591
train_loss,1.12099
val_acc,0.52


In [12]:
EPOCHS = 5
LR = 1e-4

model_pt = timm.create_model("vit_tiny_patch16_224", pretrained=True,
                             num_classes=10, in_chans=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_pt.parameters(), lr=LR)

run = wandb.init(project="tiny-wfm-backbone", name="stage-b-vit-cifar10-pretrained",
                 config={"model": "vit_tiny_patch16_224", "pretrained": True,
                         "lr": LR, "epochs": EPOCHS})

def evaluate():
    model_pt.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model_pt(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(EPOCHS):
    model_pt.train()
    running_loss = correct = total = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        logits = model_pt(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total
    val_acc    = evaluate()
    wandb.log({"epoch": epoch, "train_loss": train_loss,
               "train_acc": train_acc, "val_acc": val_acc})
    print(f"epoch {epoch}: loss {train_loss:.3f} | train {train_acc:.1%} | val {val_acc:.1%}")

wandb.finish()



epoch 0: loss 0.518 | train 82.5% | val 92.8%
epoch 1: loss 0.106 | train 96.6% | val 94.0%
epoch 2: loss 0.038 | train 98.8% | val 92.9%
epoch 3: loss 0.034 | train 98.8% | val 93.2%
epoch 4: loss 0.024 | train 99.3% | val 93.1%


epoch,▁▃▅▆█
train_acc,▁▇███
train_loss,█▂▁▁▁
val_acc,▁█▁▃▃
epoch,4
train_acc,0.9933
train_loss,0.02402
val_acc,0.931
